# Diagnose HHP Infeasibility

## Section A: Single-case diagnosis

This notebook diagnoses infeasible results in cached Experiment 6 outputs by replaying the same `(dwelling_id, run)` seeds and running A/B relaxations:

- baseline constraints
- EV-target relaxation
- thermal-capacity/comfort relaxation
- both relaxations together
- EV charging-capacity lift (e.g., 10 kW)
- monovalent HP-capacity upper-limit lift (e.g., up to 20 kW)

The output classifies each failing pair as EV-dominant, thermal-dominant, mixed, or structural/data issue.

Section A (single-case diagnosis) and Section B (batch infeasible-run update) are standalone and can be run independently.

In [ ]:
import re
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve repository root from current notebook location.
repo_root = Path.cwd().resolve()
if not (repo_root / '.git').exists():
    for parent in repo_root.parents:
        if (parent / '.git').exists():
            repo_root = parent
            break

source_dir = repo_root / 'Codes' / 'sourcecode'
source_dir_str = str(source_dir)
if source_dir_str in sys.path:
    sys.path.remove(source_dir_str)
sys.path.insert(0, source_dir_str)

import stochastic_baseload_multiple_building_simulation_and_aggregation as sbm
sbm = importlib.reload(sbm)

prepare_workflow_inputs = sbm.prepare_workflow_inputs
run_monte_carlo_batch = sbm.run_monte_carlo_batch


In [ ]:
# === User configuration ===
# Point to one cached Experiment 6 folder.
diag_case_dir = (
    Path(repo_root)
    / 'Output Data'
    / 'Single Dwelling Runs'
    / 'randomized offset'
    / 'agile_monovalent_EV_5kW_offset0p0h'
)

# Set to an int to debug quickly on a subset of failing pairs.
diag_max_pairs = None

# Workflow setup matching FullEnergyOptimizationDemo11 defaults.
workflow_cfg = {
    'step': '30min',
    'start_date': pd.Timestamp('2021-02-07'),
    'n_days': 7,
    'tariff_type': 'agile',
    'max_dwellings': None,
    'random_seed': 42,
    'include_flex_setpoint': False,
}

# Base optimizer/EV/HW settings used for replay.
optim_params_cfg = {
    'tol': 1.0,
    'COP': 3.5,
    'etaB': 0.9,
    'T0': 21.0,
}
ev_params_cfg = {
    'ev_capacity': 60.0,
    'ev_soc_init': 48.0,
    'ev_target': 48.0,
    'ev_charge_max': 5.0,
    'eta_ev_charge': 0.95,
    'ev_min_final_fraction': 0.8,
    'ev_retention': 0.999,
}
mhp_hw_params_cfg = {
    'hw_mode': 'hp_storage',
    'V_stor': 0.2,
    'V_stor_init': 0.12,
    'T_mains': 10.0,
    'T_hw_supply': 40.0,
    'Q_bo_hw_max': 0.0,
}
hhp_hw_params_cfg = {
    'hw_mode': 'boiler_only',
    'Q_hp_hw_max': 0.0,
}

# Additional capacity-lift relaxations requested by user.
diag_relax_ev_charge_max_kw = 10.0
diag_relax_monovalent_hp_max_kw = 20.0

# Relaxation execution mode:
# - 'ab_only': original A/B relaxations only
# - 'capacity_only': new capacity-limit relaxations only
# - 'both': run both paths
diag_relaxation_mode = 'both'

if not diag_case_dir.exists():
    raise FileNotFoundError(f'Case folder not found: {diag_case_dir}')

m = re.match(
    r'^(?P<tariff>[A-Za-z0-9]+)_(?P<case>hybrid|monovalent)_EV_(?P<ev>\d+)kW_offset(?P<off>[0-9p]+)h$',
    diag_case_dir.name,
)
if m:
    diag_tariff_type = m.group('tariff').lower()
    diag_case = m.group('case').lower()
    diag_ev_kw = float(m.group('ev'))
    diag_offset_hours = float(m.group('off').replace('p', '.'))
else:
    diag_tariff_type = str(workflow_cfg['tariff_type']).lower()
    diag_case = 'hybrid'
    diag_ev_kw = float(ev_params_cfg['ev_charge_max'])
    diag_offset_hours = 0.0

print('Case dir:', diag_case_dir)
print('Parsed:', {'tariff': diag_tariff_type, 'case': diag_case, 'ev_kw': diag_ev_kw, 'offset_h': diag_offset_hours})


In [ ]:
# Build context and align tariff to the diagnostic case metadata.
data_root = repo_root / 'Input data'
meta_path = data_root / 'Newcastle_Urban_Case_meta_updated.csv'
weather_path = data_root / 'NEcase_20_21_t2m_ssrd_30min.csv'
profile_root = data_root / 'Stochastic_Demands'

context = prepare_workflow_inputs(
    repo_root=repo_root,
    data_root=data_root,
    meta_path=meta_path,
    weather_path=weather_path,
    profile_root=profile_root,
    **workflow_cfg,
)

diag_context = dict(context)
diag_tariff = sbm.build_tariff(
    diag_context['window'].index[0],
    n_days=diag_context['n_days'],
    step=diag_context['step'],
    type=diag_tariff_type,
)
if not diag_tariff.index.equals(diag_context['window'].index):
    diag_tariff = diag_tariff.reindex(diag_context['window'].index, method='ffill')
diag_context['tariff'] = diag_tariff
diag_context['n_steps'] = len(diag_tariff)

diag_tariff_offset_cfg = {
    'enabled': True,
    'max_offset_hours': float(diag_offset_hours),
    'seed_base': 42,
}

active_hw_cfg = dict(hhp_hw_params_cfg) if diag_case == 'hybrid' else dict(mhp_hw_params_cfg)
active_ev_cfg = dict(ev_params_cfg)
active_ev_cfg['ev_charge_max'] = float(diag_ev_kw)

print(f"Prepared context: dwellings={len(diag_context['dwelling_inputs'])}, n_days={diag_context['n_days']}, case={diag_case}")


In [ ]:
def load_failing_pairs(case_dir: Path, max_pairs: int | None = None) -> pd.DataFrame:
    rows = []
    for fp in sorted(case_dir.glob('dwelling_*_runs_breakdown.csv')):
        m = re.match(r'^dwelling_(.+)_runs_breakdown$', fp.stem)
        if not m:
            continue
        dwelling_token = m.group(1)
        try:
            dwelling_id = int(dwelling_token)
        except Exception:
            dwelling_id = dwelling_token

        df = pd.read_csv(fp, usecols=['run', 'solve_status'])
        bad_runs = (
            df.loc[df['solve_status'].astype(str).str.lower().eq('infeasible'), 'run']
            .dropna()
            .astype(int)
            .unique()
        )
        for run_no in sorted(bad_runs.tolist()):
            rows.append({'dwelling_id': dwelling_id, 'run': int(run_no), 'source_file': str(fp)})

    out = pd.DataFrame(rows).sort_values(['dwelling_id', 'run']).reset_index(drop=True)
    if max_pairs is not None and len(out) > int(max_pairs):
        out = out.head(int(max_pairs)).copy()
    return out

diag_pairs_df = load_failing_pairs(diag_case_dir, max_pairs=diag_max_pairs)
if diag_pairs_df.empty:
    raise ValueError(f'No infeasible pairs found in {diag_case_dir}')

print(f'Failing pairs: {len(diag_pairs_df)}')
display(diag_pairs_df.head(20))


In [ ]:
def _extract_single_row(mc_results: dict) -> dict:
    frames = mc_results.get('summary_runs', [])
    for frame in frames:
        if isinstance(frame, pd.DataFrame) and not frame.empty:
            row = frame.iloc[0].to_dict()
            return {
                'status': str(row.get('solve_status', 'unknown')).lower(),
                'hp_capacity_kw': row.get('hp_capacity_kw', np.nan),
            }
    return {'status': 'unknown', 'hp_capacity_kw': np.nan}

def rerun_pair_status(
    dwelling_id,
    run_no: int,
    *,
    ev_override: dict | None = None,
    opt_override: dict | None = None,
    hw_override: dict | None = None,
    capacity_candidates_kw: np.ndarray | list[float] | None = None,
) -> tuple[str, dict, float]:
    ev_cfg = dict(active_ev_cfg)
    if ev_override:
        ev_cfg.update(ev_override)

    opt_cfg = dict(optim_params_cfg)
    if opt_override:
        opt_cfg.update(opt_override)

    hw_cfg = dict(active_hw_cfg)
    if hw_override:
        hw_cfg.update(hw_override)

    out = run_monte_carlo_batch(
        diag_context,
        mc_runs=1,
        run_index_start=int(run_no) - 1,
        case=diag_case,
        output_subdir='Diagnostics_no_save',
        selected_dwellings=[dwelling_id],
        capacity_candidates_kw=capacity_candidates_kw,
        optim_params_cfg=opt_cfg,
        ev_params_cfg=ev_cfg,
        hw_params_cfg=hw_cfg,
        tariff_random_offset_cfg=diag_tariff_offset_cfg,
        single_dwelling_id=dwelling_id,
        day_ahead=True,
        save_outputs=False,
        show_progress=False,
    )
    extracted = _extract_single_row(out)
    hp_cap = extracted['hp_capacity_kw']
    if hp_cap is None:
        hp_cap = np.nan
    return extracted['status'], out.get('failure_reasons', {}), float(hp_cap) if pd.notna(hp_cap) else np.nan

# Quick smoke test on first failing pair.
first_pair = diag_pairs_df.iloc[0]
s0, fr0, cap0 = rerun_pair_status(first_pair['dwelling_id'], int(first_pair['run']))
print('Smoke test status:', s0)
print('Smoke test hp_capacity_kw:', cap0)
if fr0:
    print('Failure reasons:', fr0)


In [ ]:
def classify_ab(base: str, ev_relaxed: str, thermal_relaxed: str, both_relaxed: str) -> str:
    b = str(base).lower()
    e = str(ev_relaxed).lower()
    t = str(thermal_relaxed).lower()
    bt = str(both_relaxed).lower()

    if b == 'optimal':
        return 'baseline_optimal'
    if e == 'optimal' and t != 'optimal':
        return 'ev_constraint_dominant'
    if t == 'optimal' and e != 'optimal':
        return 'thermal_constraint_dominant'
    if bt == 'optimal' and e != 'optimal' and t != 'optimal':
        return 'mixed_ev_and_thermal'
    if bt == 'optimal' and e == 'optimal' and t == 'optimal':
        return 'both_single_relaxations_fix'
    if bt != 'optimal':
        return 'structural_or_data_issue'
    return 'other'

mode = str(diag_relaxation_mode).strip().lower()
if mode not in {'ab_only', 'capacity_only', 'both'}:
    raise ValueError("diag_relaxation_mode must be one of: 'ab_only', 'capacity_only', 'both'")
run_ab = mode in {'ab_only', 'both'}
run_capacity = mode in {'capacity_only', 'both'}

# Original A/B relaxations used for diagnosis.
ev_relax = {
    'ev_target': 0.0,
    'ev_min_final_fraction': 0.0,
}
thermal_relax = {
    'Qbo_max': 45_000.0,
    'tol': 2.0,
}

# Additional capacity-limit relaxations requested by user.
ev_capacity_relax = {
    'ev_charge_max': float(diag_relax_ev_charge_max_kw),
}
monovalent_capacity_candidates_relax = np.arange(7.0, float(diag_relax_monovalent_hp_max_kw) + 0.5, 0.5)

diag_rows = []
for _, row in diag_pairs_df.iterrows():
    d = row['dwelling_id']
    r = int(row['run'])

    # Baseline is always evaluated as the reference point.
    base_status, base_fail, base_cap = rerun_pair_status(d, r)

    if run_ab:
        ev_status, ev_fail, ev_cap_ab = rerun_pair_status(d, r, ev_override=ev_relax)
        th_status, th_fail, th_cap_ab = rerun_pair_status(d, r, opt_override=thermal_relax)
        both_status, both_fail, both_cap_ab = rerun_pair_status(d, r, ev_override=ev_relax, opt_override=thermal_relax)
        ab_classification = classify_ab(base_status, ev_status, th_status, both_status)
    else:
        ev_status, ev_fail, ev_cap_ab = 'not_run', {}, np.nan
        th_status, th_fail, th_cap_ab = 'not_run', {}, np.nan
        both_status, both_fail, both_cap_ab = 'not_run', {}, np.nan
        ab_classification = 'not_run'

    if run_capacity:
        ev_cap_status, ev_cap_fail, ev_cap_capacity = rerun_pair_status(d, r, ev_override=ev_capacity_relax)
        if str(diag_case).lower() == 'monovalent':
            th_cap_status, th_cap_fail, th_cap_capacity = rerun_pair_status(
                d,
                r,
                capacity_candidates_kw=monovalent_capacity_candidates_relax,
            )
            both_cap_status, both_cap_fail, both_cap_capacity = rerun_pair_status(
                d,
                r,
                ev_override=ev_capacity_relax,
                capacity_candidates_kw=monovalent_capacity_candidates_relax,
            )
        else:
            # HP-capacity sweep is only used by monovalent case in this workflow.
            th_cap_status, th_cap_fail, th_cap_capacity = 'not_applicable', {}, np.nan
            both_cap_status, both_cap_fail, both_cap_capacity = ev_cap_status, ev_cap_fail, ev_cap_capacity
    else:
        ev_cap_status, ev_cap_fail, ev_cap_capacity = 'not_run', {}, np.nan
        th_cap_status, th_cap_fail, th_cap_capacity = 'not_run', {}, np.nan
        both_cap_status, both_cap_fail, both_cap_capacity = 'not_run', {}, np.nan

    diag_rows.append(
        {
            'dwelling_id': d,
            'run': r,
            'baseline_status': base_status,
            'baseline_hp_capacity_kw': base_cap,
            'ev_relaxed_status': ev_status,
            'ev_relaxed_hp_capacity_kw': ev_cap_ab,
            'thermal_relaxed_status': th_status,
            'thermal_relaxed_hp_capacity_kw': th_cap_ab,
            'both_relaxed_status': both_status,
            'both_relaxed_hp_capacity_kw': both_cap_ab,
            'classification': ab_classification,
            'baseline_failure_reasons': base_fail,
            'ev_relaxed_failure_reasons': ev_fail,
            'thermal_relaxed_failure_reasons': th_fail,
            'both_relaxed_failure_reasons': both_fail,
            'ev_capacity_relaxed_status': ev_cap_status,
            'ev_capacity_relaxed_hp_capacity_kw': ev_cap_capacity,
            'thermal_capacity_relaxed_status': th_cap_status,
            'thermal_capacity_relaxed_hp_capacity_kw': th_cap_capacity,
            'both_capacity_relaxed_status': both_cap_status,
            'both_capacity_relaxed_hp_capacity_kw': both_cap_capacity,
            'ev_capacity_relaxed_failure_reasons': ev_cap_fail,
            'thermal_capacity_relaxed_failure_reasons': th_cap_fail,
            'both_capacity_relaxed_failure_reasons': both_cap_fail,
        }
    )

diag_result_df = pd.DataFrame(diag_rows).sort_values(['classification', 'dwelling_id', 'run']).reset_index(drop=True)
display(diag_result_df)

if run_ab:
    print('Classification counts:')
    display(diag_result_df['classification'].value_counts(dropna=False).rename_axis('classification').to_frame('count'))
else:
    print('Classification counts skipped (diag_relaxation_mode does not include A/B path).')

if run_capacity:
    print('Capacity-relaxation status counts:')
    capacity_summary_df = pd.DataFrame(
        {
            'ev_capacity_relaxed': diag_result_df['ev_capacity_relaxed_status'].value_counts(dropna=False),
            'thermal_capacity_relaxed': diag_result_df['thermal_capacity_relaxed_status'].value_counts(dropna=False),
            'both_capacity_relaxed': diag_result_df['both_capacity_relaxed_status'].value_counts(dropna=False),
        }
    ).fillna(0).astype(int)
    display(capacity_summary_df)
else:
    capacity_summary_df = pd.DataFrame()
    print('Capacity-relaxation summary skipped (diag_relaxation_mode does not include capacity path).')


In [ ]:
# Save diagnostic tables next to the analyzed case folder.
diag_out_csv = diag_case_dir / 'diagnosis_ab_test_summary.csv'
diag_result_df.to_csv(diag_out_csv, index=False)
print('Saved:', diag_out_csv)

if not capacity_summary_df.empty:
    capacity_out_csv = diag_case_dir / 'diagnosis_capacity_relaxation_summary.csv'
    capacity_summary_df.to_csv(capacity_out_csv)
    print('Saved:', capacity_out_csv)
else:
    print('Capacity summary not saved (capacity path not selected).')


## Section B: Batch update for infeasible runs (all randomized-offset cases)

This section only identifies historical infeasible runs and checks whether they become feasible under the new capacity-lift strategy (EV to 10kW and monovalent HP scan up to 20kW).


In [2]:
# === Batch update for infeasible runs across all case folders ===
# Standalone setup so Section B can run without Section A.
import re
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / '.git').exists():
    for parent in repo_root.parents:
        if (parent / '.git').exists():
            repo_root = parent
            break

batch_root_dir = Path(repo_root) / 'Output Data' / 'Single Dwelling Runs' / 'randomized offset'
batch_case_pattern = '*_EV_*kW_offset*h'

# Capacity-lift strategy requested by user.
batch_refresh_ev_charge_max_kw = 5.0
batch_refresh_monovalent_hp_max_kw = 20.0
batch_refresh_hp_step_kw = 0.5

# Optional debug limit per case (None = all infeasible runs).
batch_max_pairs_per_case = None
batch_write_case_csv = True

case_name_regex = re.compile(r'^(?P<tariff>[A-Za-z0-9]+)_(?P<case>hybrid|monovalent)_EV_(?P<ev>\d+)kW_offset(?P<off>[0-9p]+)h$')

def parse_case_folder(case_dir: Path) -> dict:
    m = case_name_regex.match(case_dir.name)
    if not m:
        raise ValueError(f'Case folder name does not match expected pattern: {case_dir.name}')
    return {
        'case_name': case_dir.name,
        'case_dir': str(case_dir),
        'tariff_type': m.group('tariff').lower(),
        'case_type': m.group('case').lower(),
        'ev_kw': float(m.group('ev')),
        'offset_hours': float(m.group('off').replace('p', '.')),
    }

def load_infeasible_pairs_for_case(case_dir: Path, max_pairs: int | None = None) -> pd.DataFrame:
    case_meta = parse_case_folder(case_dir)
    rows = []
    for fp in sorted(case_dir.glob('dwelling_*_runs_breakdown.csv')):
        m = re.match(r'^dwelling_(.+)_runs_breakdown$', fp.stem)
        if not m:
            continue
        dwelling_token = m.group(1)
        try:
            dwelling_id = int(dwelling_token)
        except Exception:
            dwelling_id = dwelling_token

        df = pd.read_csv(fp, usecols=['run', 'solve_status'])
        infeasible_runs = (
            df.loc[df['solve_status'].astype(str).str.lower().eq('infeasible'), 'run']
            .dropna()
            .astype(int)
            .unique()
        )
        for run_no in sorted(infeasible_runs.tolist()):
            rows.append({
                **case_meta,
                'dwelling_id': dwelling_id,
                'run': int(run_no),
                'source_file': str(fp),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=['case_name', 'case_dir', 'tariff_type', 'case_type', 'ev_kw', 'offset_hours', 'dwelling_id', 'run', 'source_file'])
    out = out.sort_values(['case_name', 'dwelling_id', 'run']).reset_index(drop=True)
    if max_pairs is not None and len(out) > int(max_pairs):
        out = out.head(int(max_pairs)).copy()
    return out

case_dirs = sorted([p for p in batch_root_dir.glob(batch_case_pattern) if p.is_dir()])
if not case_dirs:
    raise FileNotFoundError(f'No case folders found under: {batch_root_dir}')

infeasible_frames = []
skipped_case_dirs = []
for case_dir in case_dirs:
    try:
        case_df = load_infeasible_pairs_for_case(case_dir, max_pairs=batch_max_pairs_per_case)
    except Exception as exc:
        skipped_case_dirs.append({
            'case_dir': str(case_dir),
            'error': str(exc),
        })
        continue
    if not case_df.empty:
        infeasible_frames.append(case_df)

if infeasible_frames:
    infeasible_before_df = pd.concat(infeasible_frames, ignore_index=True)
else:
    infeasible_before_df = pd.DataFrame(columns=['case_name', 'case_dir', 'tariff_type', 'case_type', 'ev_kw', 'offset_hours', 'dwelling_id', 'run', 'source_file'])

if skipped_case_dirs:
    skipped_cases_df = pd.DataFrame(skipped_case_dirs)
    print('Skipped case folders due to parsing/IO errors:')
    display(skipped_cases_df)
else:
    skipped_cases_df = pd.DataFrame(columns=['case_dir', 'error'])

print('Detected case folders:', len(case_dirs))
print('Previously infeasible runs found:', len(infeasible_before_df))

if not infeasible_before_df.empty:
    infeasible_case_counts_df = (
        infeasible_before_df.groupby(['case_name', 'case_type'], as_index=False)
        .size()
        .rename(columns={'size': 'previously_infeasible_runs'})
        .sort_values(['case_name'])
        .reset_index(drop=True)
    )
else:
    infeasible_case_counts_df = pd.DataFrame(columns=['case_name', 'case_type', 'previously_infeasible_runs'])

display(infeasible_case_counts_df)
display(infeasible_before_df.head(50))


Detected case folders: 12
Previously infeasible runs found: 264


,case_name,case_type,previously_infeasible_runs
0,agile_hybrid_EV_5kW_offset0p0h,hybrid,22
1,agile_hybrid_EV_5kW_offset0p5h,hybrid,22
2,agile_hybrid_EV_5kW_offset1p0h,hybrid,22
3,agile_hybrid_EV_5kW_offset2p0h,hybrid,22
4,agile_hybrid_EV_5kW_offset3p0h,hybrid,22
5,agile_hybrid_EV_5kW_offset4p0h,hybrid,22
6,agile_monovalent_EV_5kW_offset0p0h,monovalent,22
7,agile_monovalent_EV_5kW_offset0p5h,monovalent,22
8,agile_monovalent_EV_5kW_offset1p0h,monovalent,22
9,agile_monovalent_EV_5kW_offset2p0h,monovalent,22


,case_name,case_dir,tariff_type,case_type,ev_kw,offset_hours,dwelling_id,run,source_file
0,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,4,6,E:\GitHubProjects\LV network\Output Data\Singl...
1,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,6,10,E:\GitHubProjects\LV network\Output Data\Singl...
2,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,12,7,E:\GitHubProjects\LV network\Output Data\Singl...
3,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,22,7,E:\GitHubProjects\LV network\Output Data\Singl...
4,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,29,3,E:\GitHubProjects\LV network\Output Data\Singl...
5,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,43,1,E:\GitHubProjects\LV network\Output Data\Singl...
6,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,55,6,E:\GitHubProjects\LV network\Output Data\Singl...
7,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,56,7,E:\GitHubProjects\LV network\Output Data\Singl...
8,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,56,10,E:\GitHubProjects\LV network\Output Data\Singl...
9,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,66,6,E:\GitHubProjects\LV network\Output Data\Singl...


In [2]:
# Recheck historical infeasible runs under the new capacity-lift strategy.
# Standalone setup so Section B can run without Section A.
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

if 'repo_root' not in globals():
    repo_root = Path.cwd().resolve()
    if not (repo_root / '.git').exists():
        for parent in repo_root.parents:
            if (parent / '.git').exists():
                repo_root = parent
                break

source_dir = repo_root / 'Codes' / 'sourcecode'
source_dir_str = str(source_dir)
if source_dir_str in sys.path:
    sys.path.remove(source_dir_str)
sys.path.insert(0, source_dir_str)

import stochastic_baseload_multiple_building_simulation_and_aggregation as sbm
sbm = importlib.reload(sbm)
prepare_workflow_inputs = sbm.prepare_workflow_inputs
run_monte_carlo_batch = sbm.run_monte_carlo_batch

if 'workflow_cfg' not in globals():
    workflow_cfg = {
        'step': '30min',
        'start_date': pd.Timestamp('2021-02-07'),
        'n_days': 7,
        'tariff_type': 'agile',
        'max_dwellings': None,
        'random_seed': 42,
        'include_flex_setpoint': False,
    }
if 'optim_params_cfg' not in globals():
    optim_params_cfg = {
        'tol': 1.0,
        'COP': 3.5,
        'etaB': 0.9,
        'T0': 21.0,
    }
if 'ev_params_cfg' not in globals():
    ev_params_cfg = {
        'ev_capacity': 60.0,
        'ev_soc_init': 48.0,
        'ev_target': 48.0,
        'ev_charge_max': 5.0,
        'eta_ev_charge': 0.95,
        'ev_min_final_fraction': 0.8,
        'ev_retention': 0.999,
    }
if 'mhp_hw_params_cfg' not in globals():
    mhp_hw_params_cfg = {
        'hw_mode': 'hp_storage',
        'V_stor': 0.2,
        'V_stor_init': 0.12,
        'T_mains': 10.0,
        'T_hw_supply': 40.0,
        'Q_bo_hw_max': 0.0,
    }
if 'hhp_hw_params_cfg' not in globals():
    hhp_hw_params_cfg = {
        'hw_mode': 'boiler_only',
        'Q_hp_hw_max': 0.0,
    }
if 'batch_refresh_ev_charge_max_kw' not in globals():
    batch_refresh_ev_charge_max_kw = 10.0
if 'batch_refresh_monovalent_hp_max_kw' not in globals():
    batch_refresh_monovalent_hp_max_kw = 20.0
if 'batch_refresh_hp_step_kw' not in globals():
    batch_refresh_hp_step_kw = 0.5

if 'context' not in globals():
    data_root = repo_root / 'Input data'
    meta_path = data_root / 'Newcastle_Urban_Case_meta_updated.csv'
    weather_path = data_root / 'NEcase_20_21_t2m_ssrd_30min.csv'
    profile_root = data_root / 'Stochastic_Demands'
    context = prepare_workflow_inputs(
        repo_root=repo_root,
        data_root=data_root,
        meta_path=meta_path,
        weather_path=weather_path,
        profile_root=profile_root,
        **workflow_cfg,
    )

if '_extract_single_row' not in globals():
    def _extract_single_row(mc_results: dict) -> dict:
        frames = mc_results.get('summary_runs', [])
        for frame in frames:
            if isinstance(frame, pd.DataFrame) and not frame.empty:
                row = frame.iloc[0].to_dict()
                return {
                    'status': str(row.get('solve_status', 'unknown')).lower(),
                    'hp_capacity_kw': row.get('hp_capacity_kw', np.nan),
                }
        return {'status': 'unknown', 'hp_capacity_kw': np.nan}

if 'infeasible_before_df' not in globals():
    raise NameError('Run Section B infeasible-list block before this recheck block.')

tariff_context_cache: dict[str, dict] = {}

def build_case_context_for_tariff(tariff_type: str) -> dict:
    if tariff_type in tariff_context_cache:
        return tariff_context_cache[tariff_type]

    case_context = dict(context)
    tdf = sbm.build_tariff(
        context['window'].index[0],
        n_days=context['n_days'],
        step=context['step'],
        type=tariff_type,
    )
    if not tdf.index.equals(context['window'].index):
        tdf = tdf.reindex(context['window'].index, method='ffill')
    case_context['tariff'] = tdf
    case_context['n_steps'] = len(tdf)
    tariff_context_cache[tariff_type] = case_context
    return case_context

def rerun_with_capacity_lift(row: pd.Series) -> dict:
    case_type = str(row['case_type']).lower()
    tariff_type = str(row['tariff_type']).lower()
    case_context = build_case_context_for_tariff(tariff_type)

    dwelling_id = row['dwelling_id']
    try:
        dwelling_id = int(dwelling_id)
    except Exception:
        pass

    ev_cfg = dict(ev_params_cfg)
    ev_cfg['ev_charge_max'] = max(float(row['ev_kw']), float(batch_refresh_ev_charge_max_kw))

    if case_type == "hybrid":
        hw_cfg = dict(hhp_hw_params_cfg)
        capacity_candidates = None
    else:
        hw_cfg = dict(mhp_hw_params_cfg)
        capacity_candidates = np.arange(
            7.0,
            float(batch_refresh_monovalent_hp_max_kw) + float(batch_refresh_hp_step_kw) / 2.0,
            float(batch_refresh_hp_step_kw),
        )

    tariff_random_offset_cfg = {
        'enabled': True,
        'max_offset_hours': float(row['offset_hours']),
        'seed_base': 42,
    }

    out = run_monte_carlo_batch(
        case_context,
        mc_runs=1,
        run_index_start=int(row['run']) - 1,
        case=case_type,
        output_subdir='Diagnostics_no_save',
        selected_dwellings=[dwelling_id],
        capacity_candidates_kw=capacity_candidates,
        optim_params_cfg=optim_params_cfg,
        ev_params_cfg=ev_cfg,
        hw_params_cfg=hw_cfg,
        tariff_random_offset_cfg=tariff_random_offset_cfg,
        single_dwelling_id=dwelling_id,
        day_ahead=True,
        save_outputs=False,
        show_progress=False,
    )

    extracted = _extract_single_row(out)
    status = str(extracted.get('status', 'unknown')).lower()
    hp_capacity = extracted.get('hp_capacity_kw', np.nan)
    if hp_capacity is None:
        hp_capacity = np.nan
    return {
        'updated_status': status,
        'became_feasible': status == 'optimal',
        'feasible_hp_capacity_kw': float(hp_capacity) if pd.notna(hp_capacity) else np.nan,
        'failure_reasons': out.get('failure_reasons', {}),
    }

if infeasible_before_df.empty:
    refresh_results_df = infeasible_before_df.copy()
    refresh_results_df['updated_status'] = pd.Series(dtype='object')
    refresh_results_df['became_feasible'] = pd.Series(dtype='bool')
    refresh_results_df['feasible_hp_capacity_kw'] = pd.Series(dtype='float64')
    refresh_results_df['failure_reasons'] = pd.Series(dtype='object')
else:
    refresh_rows = []
    try:
        from tqdm.auto import tqdm
    except Exception:
        def tqdm(x, **kwargs):
            return x
    row_iter = tqdm(
        infeasible_before_df.iterrows(),
        total=len(infeasible_before_df),
        desc='Rechecking infeasible runs',
    )
    for _, base_row in row_iter:
        update_info = rerun_with_capacity_lift(base_row)
        merged = dict(base_row.to_dict())
        merged.update(update_info)
        refresh_rows.append(merged)
    refresh_results_df = pd.DataFrame(refresh_rows)

if not refresh_results_df.empty:
    refresh_case_summary_df = (
        refresh_results_df.groupby(['case_name', 'case_type'], as_index=False)
        .agg(
            previously_infeasible_runs=('run', 'size'),
            became_feasible_runs=('became_feasible', 'sum'),
        )
    )
    refresh_case_summary_df['still_infeasible_runs'] = (
        refresh_case_summary_df['previously_infeasible_runs'] - refresh_case_summary_df['became_feasible_runs']
    )
    refresh_case_summary_df['recovered_ratio'] = (
        refresh_case_summary_df['became_feasible_runs'] / refresh_case_summary_df['previously_infeasible_runs']
    )
else:
    refresh_case_summary_df = pd.DataFrame(
        columns=['case_name', 'case_type', 'previously_infeasible_runs', 'became_feasible_runs', 'still_infeasible_runs', 'recovered_ratio']
    )

print('Recheck rows:', len(refresh_results_df))
display(refresh_case_summary_df.sort_values(['case_name']).reset_index(drop=True))
display(refresh_results_df.head(50))


Rechecking infeasible runs:   0%|          | 0/1474 [00:00<?, ?it/s]

Set parameter Username
Academic license - for non-commercial use only - expires 2027-03-18
Recheck rows: 1474


,case_name,case_type,previously_infeasible_runs,became_feasible_runs,still_infeasible_runs,recovered_ratio
0,agile_hybrid_EV_5kW_offset0p0h,hybrid,22,0,22,0.000000
1,agile_hybrid_EV_5kW_offset0p5h,hybrid,22,0,22,0.000000
2,agile_hybrid_EV_5kW_offset1p0h,hybrid,22,0,22,0.000000
3,agile_hybrid_EV_5kW_offset2p0h,hybrid,22,0,22,0.000000
4,agile_hybrid_EV_5kW_offset3p0h,hybrid,22,0,22,0.000000
5,agile_hybrid_EV_5kW_offset4p0h,hybrid,22,0,22,0.000000
6,agile_monovalent_EV_5kW_offset0p0h,monovalent,112,90,22,0.803571
7,agile_monovalent_EV_5kW_offset0p5h,monovalent,112,90,22,0.803571
8,agile_monovalent_EV_5kW_offset1p0h,monovalent,112,90,22,0.803571
9,agile_monovalent_EV_5kW_offset2p0h,monovalent,112,90,22,0.803571


,case_name,case_dir,tariff_type,case_type,ev_kw,offset_hours,dwelling_id,run,source_file,updated_status,became_feasible,feasible_hp_capacity_kw,failure_reasons
0,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,4,6,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
1,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,6,10,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
2,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,12,7,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
3,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,22,7,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
4,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,29,3,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
5,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,43,1,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
6,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,55,6,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
7,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,56,7,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
8,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,56,10,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}
9,agile_hybrid_EV_5kW_offset0p0h,E:\GitHubProjects\LV network\Output Data\Singl...,agile,hybrid,5.0,0.0,66,6,E:\GitHubProjects\LV network\Output Data\Singl...,infeasible,False,NaN,{'Gurobi failed on full problem. Status 3': 1}


In [3]:
# Save outputs for full list and per-case tracking.
batch_infeasible_list_csv = batch_root_dir / 'infeasible_runs_before_update.csv'
batch_refresh_detail_csv = batch_root_dir / 'infeasible_runs_after_capacity_update.csv'
batch_refresh_case_summary_csv = batch_root_dir / 'infeasible_runs_case_summary_after_capacity_update.csv'

infeasible_before_df.to_csv(batch_infeasible_list_csv, index=False)
refresh_results_df.to_csv(batch_refresh_detail_csv, index=False)
refresh_case_summary_df.to_csv(batch_refresh_case_summary_csv, index=False)

print('Saved:', batch_infeasible_list_csv)
print('Saved:', batch_refresh_detail_csv)
print('Saved:', batch_refresh_case_summary_csv)

if batch_write_case_csv and not refresh_results_df.empty:
    for case_name, gdf in refresh_results_df.groupby('case_name', sort=True):
        case_out_csv = batch_root_dir / case_name / 'infeasible_runs_after_capacity_update.csv'
        gdf.to_csv(case_out_csv, index=False)
        print('Saved:', case_out_csv)
elif batch_write_case_csv:
    print('Per-case CSV skipped (no refresh rows).')


Saved: E:\GitHubProjects\LV network\Output Data\Single Dwelling Runs\randomized offset\infeasible_runs_before_update.csv
Saved: E:\GitHubProjects\LV network\Output Data\Single Dwelling Runs\randomized offset\infeasible_runs_after_capacity_update.csv
Saved: E:\GitHubProjects\LV network\Output Data\Single Dwelling Runs\randomized offset\infeasible_runs_case_summary_after_capacity_update.csv
Saved: E:\GitHubProjects\LV network\Output Data\Single Dwelling Runs\randomized offset\agile_hybrid_EV_5kW_offset0p0h\infeasible_runs_after_capacity_update.csv
Saved: E:\GitHubProjects\LV network\Output Data\Single Dwelling Runs\randomized offset\agile_hybrid_EV_5kW_offset0p5h\infeasible_runs_after_capacity_update.csv
Saved: E:\GitHubProjects\LV network\Output Data\Single Dwelling Runs\randomized offset\agile_hybrid_EV_5kW_offset1p0h\infeasible_runs_after_capacity_update.csv
Saved: E:\GitHubProjects\LV network\Output Data\Single Dwelling Runs\randomized offset\agile_hybrid_EV_5kW_offset2p0h\infeasible